In [1]:
import requests
import pandas as pd
import openpyxl
from dotenv import load_dotenv

print(" All packages ready — good to go!")

 All packages ready — good to go!


In [3]:
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

In [ ]:
# 8 Tech Reviewer Channels 
# Channel IDs sourced from their YouTube URLs (publicly visible)
CHANNELS = {
    "MKBHD":             "UCBJycsmduvYEL83R_U4JriQ",
    "Linus Tech Tips":   "UCXuqSBlHAE6Xw-yeJA0Tunw",
    "Dave2D":            "UCVYamHliCI9rw1tHR1xbkfw",
    "JerryRigEverything":"UCWFKCr40YwOZQx8FHU_ZqqQ",
    "Unbox Therapy":     "UCsTcErHg8oDvUnTzoqsYeNw",
    "MrMobile":          "UCSOpcUkE-is7u7c4AkLgqTw",
    "Arun Maini":        "UCddiUEpeqJcYeBxX1IVBKvQ",
    "TechLinked":        "UCeeFfhMcJa1kjtfZAGskOCA",
}


In [ ]:
# API Call: channels.list 
def get_channel_stats(channel_id, channel_name):
    url = "https://www.googleapis.com/youtube/v3/channels"
    params = {
        "part":   "snippet,statistics,contentDetails",
        "id":     channel_id,
        "key":    API_KEY,
    }
    response = requests.get(url, params=params)
    data = response.json()

    if "error" in data:
        print(f" API error for {channel_name}: {data['error']['message']}")
        return None

    if not data.get("items"):
        print(f" No data returned for {channel_name}")
        return None

    item  = data["items"][0]
    stats = item["statistics"]

    return {
        "channel_name":       channel_name,
        "channel_id":         channel_id,
        "subscribers":        int(stats.get("subscriberCount", 0)),
        "total_views":        int(stats.get("viewCount", 0)),
        "total_videos":       int(stats.get("videoCount", 0)),
        "country":            item["snippet"].get("country", "N/A"),
        "joined_date":        item["snippet"].get("publishedAt", "")[:10],
    }



In [ ]:
#  Pull all channels 
print("Pulling channel stats...\n")
rows = []

for name, cid in CHANNELS.items():
    result = get_channel_stats(cid, name)
    if result:
        rows.append(result)
        print(f"{name} — {int(result['subscribers']):,} subscribers")



Pulling channel stats...

MKBHD — 20,800,000 subscribers
Linus Tech Tips — 16,800,000 subscribers
Dave2D — 3,690,000 subscribers
JerryRigEverything — 9,910,000 subscribers
Unbox Therapy — 25,200,000 subscribers
MrMobile — 1,270,000 subscribers
Arun Maini — 3,490,000 subscribers
TechLinked — 1,990,000 subscribers


In [ ]:
#  Build DataFrame 
channels_df = pd.DataFrame(rows)
print(f"\n Done — {len(channels_df)} channels pulled")
print(channels_df[["channel_name", "subscribers", "total_views", "total_videos"]].to_string(index=False))


 Done — 8 channels pulled
      channel_name  subscribers  total_views  total_videos
             MKBHD     20800000   5315090427          1807
   Linus Tech Tips     16800000   9405597436          7689
            Dave2D      3690000    851028566           824
JerryRigEverything      9910000   2869035212          1515
     Unbox Therapy     25200000   5088987835          2490
          MrMobile      1270000    255268815           626
        Arun Maini      3490000   1126301670          5959
        TechLinked      1990000    650213921          1281


 # Pull Recent Videos Per Channel

In [ ]:
import time
import re

#  Helper: convert ISO 8601 duration → seconds 
# YouTube returns duration like "PT14M32S" — we convert to seconds
def parse_duration(iso_duration):
    if not iso_duration:
        return 0
    pattern = r'PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?'
    match = re.match(pattern, iso_duration)
    if not match:
        return 0
    hours   = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return hours * 3600 + minutes * 60 + seconds

#  Step 1: Get video IDs from a channel 
def get_video_ids(channel_id, channel_name, max_results=50):
    url = "https://www.googleapis.com/youtube/v3/search"
    params = {
        "part":       "id",
        "channelId":  channel_id,
        "maxResults": max_results,
        "order":      "date",          # most recent first
        "type":       "video",
        "key":        API_KEY,
    }
    response = requests.get(url, params=params)
    data = response.json()

    if "error" in data:
        print(f" Error fetching video IDs for {channel_name}: {data['error']['message']}")
        return []

    ids = [item["id"]["videoId"] for item in data.get("items", [])]
    print(f"  → {len(ids)} video IDs fetched")
    return ids

#  Step 2: Get stats for a batch of video IDs (max 50 per call) 
def get_video_stats(video_ids, channel_name, channel_id):
    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part":  "snippet,statistics,contentDetails",
        "id":    ",".join(video_ids),
        "key":   API_KEY,
    }
    response = requests.get(url, params=params)
    data = response.json()

    if "error" in data:
        print(f" Error fetching video stats for {channel_name}: {data['error']['message']}")
        return []

    rows = []
    for item in data.get("items", []):
        snippet  = item.get("snippet", {})
        stats    = item.get("statistics", {})
        content  = item.get("contentDetails", {})

        duration_s = parse_duration(content.get("duration", ""))

        rows.append({
            "channel_name":    channel_name,
            "channel_id":      channel_id,
            "video_id":        item["id"],
            "title":           snippet.get("title", ""),
            "published_at":    snippet.get("publishedAt", "")[:10],
            "views":           int(stats.get("viewCount",    0)),
            "likes":           int(stats.get("likeCount",    0)),
            "comments":        int(stats.get("commentCount", 0)),
            "duration_s":      duration_s,
            "duration_min":    round(duration_s / 60, 2),
            "tags":            len(snippet.get("tags", [])),
            "thumbnail":       snippet.get("thumbnails", {}).get("high", {}).get("url", ""),
        })
    return rows

#  Pull everything 
print("Pulling video data for all channels...\n")
all_videos = []

for name, cid in CHANNELS.items():
    print(f" {name}")

    # Step 1: get IDs
    video_ids = get_video_ids(cid, name, max_results=50)

    if not video_ids:
        continue

    # Step 2: get stats
    videos = get_video_stats(video_ids, name, cid)
    all_videos.extend(videos)
    print(f"  → {len(videos)} videos with stats pulled")

    time.sleep(0.5)  # be polite to the API

#  Build DataFrame 
videos_df = pd.DataFrame(all_videos)
videos_df["published_at"] = pd.to_datetime(videos_df["published_at"])

print(f"\n Total videos pulled: {len(videos_df)}")
print(videos_df[["channel_name", "title", "views", "likes", "comments", "duration_min"]].head(10).to_string(index=False))

Pulling video data for all channels...

 MKBHD
  → 17 video IDs fetched
  → 17 videos with stats pulled
 Linus Tech Tips
  → 11 video IDs fetched
  → 11 videos with stats pulled
 Dave2D
  → 50 video IDs fetched
  → 50 videos with stats pulled
 JerryRigEverything
  → 50 video IDs fetched
  → 50 videos with stats pulled
 Unbox Therapy
  → 5 video IDs fetched
  → 5 videos with stats pulled
 MrMobile
  → 50 video IDs fetched
  → 50 videos with stats pulled
 Arun Maini
  → 11 video IDs fetched
  → 11 videos with stats pulled
 TechLinked
  → 50 video IDs fetched
  → 50 videos with stats pulled

 Total videos pulled: 244
channel_name                                         title   views  likes  comments  duration_min
       MKBHD  Nothing Phone 4A/Pro Review: I Have a Theory 2143353  66439      2701         12.07
       MKBHD     Xiaomi 17 Ultra - More Camera than Phone! 3760736  96186      3781         12.08
       MKBHD Driving Xiaomi's Electric Car: Are we Cooked? 9790770 258231     25552 

Clean & Transform Data

In [ ]:
import numpy as np

print("Cleaning and transforming data...\n")

#  1. Remove duplicates & bad rows 
before = len(videos_df)
videos_df = videos_df.drop_duplicates(subset="video_id")
videos_df = videos_df[videos_df["views"] > 0]          # drop zero-view videos
videos_df = videos_df[videos_df["duration_s"] > 60]    # drop shorts (< 60s)
after = len(videos_df)
print(f" Removed {before - after} duplicate/invalid rows → {after} clean videos")

#  2. Merge channel-level stats into video table 
videos_df = videos_df.merge(
    channels_df[["channel_id", "subscribers", "total_views", "total_videos", "country", "joined_date"]],
    on="channel_id",
    how="left"
)
print(" Channel stats merged into video table")

#  3. Engineered features 

# Engagement Rate = (likes + comments) / views × 100
videos_df["engagement_rate"] = (
    (videos_df["likes"] + videos_df["comments"]) / videos_df["views"] * 100
).round(2)

# Like Rate = likes / views × 100
videos_df["like_rate"] = (
    videos_df["likes"] / videos_df["views"] * 100
).round(2)

# Comment Rate = comments / views × 100
videos_df["comment_rate"] = (
    videos_df["comments"] / videos_df["views"] * 100
).round(2)

# Views per Subscriber = how well video reaches beyond subscriber base
videos_df["views_per_sub"] = (
    videos_df["views"] / videos_df["subscribers"]
).round(4)

# Video age in days (from today)
today = pd.Timestamp.today().normalize()
videos_df["days_since_published"] = (today - videos_df["published_at"]).dt.days

# Views per day (normalizes old vs new videos)
videos_df["views_per_day"] = (
    videos_df["views"] / videos_df["days_since_published"].replace(0, 1)
).round(1)

# Duration bucket (for format analysis)
def bucket_duration(mins):
    if mins < 5:    return "Short (< 5 min)"
    elif mins < 10: return "Medium (5–10 min)"
    elif mins < 20: return "Long (10–20 min)"
    else:           return "Deep Dive (20+ min)"

videos_df["duration_bucket"] = videos_df["duration_min"].apply(bucket_duration)

# Performance tier per channel (top 25% / mid 50% / bottom 25% by views)
def assign_tier(group):
    q75 = group["views"].quantile(0.75)
    q25 = group["views"].quantile(0.25)
    conditions = [
        group["views"] >= q75,
        group["views"] <= q25,
    ]
    choices = ["Top 25%", "Bottom 25%"]
    return np.select(conditions, choices, default="Mid 50%")

videos_df["performance_tier"] = (
    videos_df.groupby("channel_name", group_keys=False)
    .apply(assign_tier)
)

# Published year + month (useful for time slicers in Power BI)
videos_df["published_year"]  = videos_df["published_at"].dt.year
videos_df["published_month"] = videos_df["published_at"].dt.month
videos_df["published_month_name"] = videos_df["published_at"].dt.strftime("%b")

print(" Engineered features added")

#  4. Benchmarks table 
# Sourced from Influencer Marketing Hub 2024 + CreatorIQ benchmarks
benchmarks = pd.DataFrame([
    {"metric": "engagement_rate", "good": 3.0,    "great": 5.0,   "unit": "%"},
    {"metric": "like_rate",       "good": 2.0,    "great": 4.0,   "unit": "%"},
    {"metric": "comment_rate",    "good": 0.05,   "great": 0.15,  "unit": "%"},
    {"metric": "views_per_sub",   "good": 0.10,   "great": 0.30,  "unit": "ratio"},
])
print(" Benchmarks table created")

#  5. Channel summary table 
channel_summary = videos_df.groupby("channel_name").agg(
    videos_analyzed   = ("video_id",         "count"),
    avg_views         = ("views",            "mean"),
    median_views      = ("views",            "median"),
    total_views       = ("views",            "sum"),
    avg_likes         = ("likes",            "mean"),
    avg_comments      = ("comments",         "mean"),
    avg_engagement_rate = ("engagement_rate","mean"),
    avg_like_rate     = ("like_rate",        "mean"),
    avg_duration_min  = ("duration_min",     "mean"),
    avg_views_per_sub = ("views_per_sub",    "mean"),
    avg_views_per_day = ("views_per_day",    "mean"),
    subscribers       = ("subscribers",      "first"),
).reset_index().round(2)

channel_summary["er_flag"] = channel_summary["avg_engagement_rate"].apply(
    lambda x: "Great" if x >= 5.0 else ("Good" if x >= 3.0 else "Below Benchmark")
)

print(" Channel summary table created")

#  6. Preview 
print("\n── Video Table Sample ──")
print(videos_df[["channel_name","title","views","engagement_rate",
                  "views_per_sub","duration_bucket","performance_tier"]
               ].head(8).to_string(index=False))

print("\n── Channel Summary ──")
print(channel_summary[["channel_name","videos_analyzed","avg_views",
                        "avg_engagement_rate","er_flag","subscribers"]
                      ].to_string(index=False))

Cleaning and transforming data...

 Removed 24 duplicate/invalid rows → 220 clean videos
 Channel stats merged into video table
 Engineered features added
 Benchmarks table created
 Channel summary table created

── Video Table Sample ──
channel_name                                         title   views  engagement_rate  views_per_sub     duration_bucket performance_tier
       MKBHD  Nothing Phone 4A/Pro Review: I Have a Theory 2143353             3.23         0.1030    Long (10–20 min)              NaN
       MKBHD     Xiaomi 17 Ultra - More Camera than Phone! 3760736             2.66         0.1808    Long (10–20 min)              NaN
       MKBHD Driving Xiaomi's Electric Car: Are we Cooked? 9790770             2.90         0.4707    Long (10–20 min)              NaN
       MKBHD              M4 Macbook Air Review: Too Easy! 6701480             2.33         0.3222   Medium (5–10 min)              NaN
       MKBHD      Google Pixel 6A Review: Can You Feel It? 4271066             3.0

C:\Users\soham\AppData\Local\Temp\ipykernel_34744\1091413268.py:74: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_tier)


In [ ]:
#  Fix performance_tier NaN issue 
def assign_tier(group):
    q75 = group["views"].quantile(0.75)
    q25 = group["views"].quantile(0.25)
    result = []
    for v in group["views"]:
        if v >= q75:
            result.append("Top 25%")
        elif v <= q25:
            result.append("Bottom 25%")
        else:
            result.append("Mid 50%")
    return pd.Series(result, index=group.index)

videos_df["performance_tier"] = (
    videos_df.groupby("channel_name")["views"]
    .transform(lambda x: assign_tier(x.to_frame("views")))
)

#  Verify fix 
null_count = videos_df["performance_tier"].isna().sum()
tier_counts = videos_df["performance_tier"].value_counts()

print(f"NaN in performance_tier: {null_count}")
print(f"\nTier distribution:\n{tier_counts.to_string()}")

#  Final data quality check 
print("\n── Null check across key columns ──")
key_cols = ["channel_name","views","likes","comments","engagement_rate",
            "views_per_sub","duration_bucket","performance_tier",
            "published_year","days_since_published"]
print(videos_df[key_cols].isnull().sum().to_string())

print(f"\n── Date range of videos ──")
print(f"Oldest: {videos_df['published_at'].min().date()}")
print(f"Newest: {videos_df['published_at'].max().date()}")

print(f"\n── Videos per channel ──")
print(videos_df["channel_name"].value_counts().to_string())

print(f"\n Final dataset: {len(videos_df)} videos | {videos_df['channel_name'].nunique()} channels")

NaN in performance_tier: 0

Tier distribution:
performance_tier
Mid 50%       106
Bottom 25%     57
Top 25%        57

── Null check across key columns ──
channel_name            0
views                   0
likes                   0
comments                0
engagement_rate         0
views_per_sub           0
duration_bucket         0
performance_tier        0
published_year          0
days_since_published    0

── Date range of videos ──
Oldest: 2014-03-26
Newest: 2026-03-24

── Videos per channel ──
channel_name
Dave2D                50
MrMobile              50
TechLinked            47
JerryRigEverything    40
MKBHD                 16
Linus Tech Tips        7
Arun Maini             6
Unbox Therapy          4

 Final dataset: 220 videos | 8 channels


Export to Excel for Power BI

In [ ]:
import os

#  Create output folder 
output_dir = "hardscope_output"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "yt_tech_reviewers.xlsx")

#  Write all 3 tables to separate sheets 
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    # Sheet 1: video-level (main fact table)
    videos_df[[
        "video_id", "channel_name", "channel_id",
        "title", "published_at", "published_year",
        "published_month", "published_month_name",
        "views", "likes", "comments",
        "engagement_rate", "like_rate", "comment_rate",
        "views_per_sub", "views_per_day",
        "duration_s", "duration_min", "duration_bucket",
        "days_since_published", "performance_tier",
        "subscribers", "thumbnail"
    ]].to_excel(writer, sheet_name="videos", index=False)

    # Sheet 2: channel summary
    channel_summary.to_excel(writer, sheet_name="channel_summary", index=False)

    # Sheet 3: benchmarks
    benchmarks.to_excel(writer, sheet_name="benchmarks", index=False)

print(f" Excel file saved → {output_path}")
print(f"\n── Sheets written ──")
print(f"  • videos          → {len(videos_df)} rows, 23 columns")
print(f"  • channel_summary → {len(channel_summary)} rows")
print(f"  • benchmarks      → {len(benchmarks)} rows")

#  Also save CSVs as backup 
videos_df.to_csv(os.path.join(output_dir, "videos.csv"), index=False)
channel_summary.to_csv(os.path.join(output_dir, "channel_summary.csv"), index=False)
benchmarks.to_csv(os.path.join(output_dir, "benchmarks.csv"), index=False)

print(f"\n CSV backups saved → {output_dir}/")
print(f"\n── File location ──")
print(f"  {os.path.abspath(output_path)}")

 Excel file saved → hardscope_output\yt_tech_reviewers.xlsx

── Sheets written ──
  • videos          → 220 rows, 23 columns
  • channel_summary → 8 rows
  • benchmarks      → 4 rows

 CSV backups saved → hardscope_output/

── File location ──
  c:\Users\soham\Downloads\HardScope assignment\hardscope_output\yt_tech_reviewers.xlsx


In [14]:
print(channel_summary[["channel_name", "avg_views_per_sub"]].sort_values("avg_views_per_sub", ascending=False).to_string(index=False))

      channel_name  avg_views_per_sub
        TechLinked               0.24
JerryRigEverything               0.22
             MKBHD               0.19
            Dave2D               0.19
          MrMobile               0.19
   Linus Tech Tips               0.05
     Unbox Therapy               0.02
        Arun Maini               0.00
